GROUP NAME: Cognitive Crew

TEAM MEMBERS:
  1. Lanka Devi Satwika - B24DS013
  2. Kotapati Sai Mounika - B24CS019
  3. Bailapudi Kusuma Teja - B24CS011
  4. Bodike Chaithali - B24CS013
  5. Jayasurya Boorada

## PS-4

# Problem Statement 4 – University Exam Timetable Optimization

## Objective

The objective is to create an examination timetable that minimizes:

1. Conflicts between courses with common students.
2. Distribution penalties caused by placing too many courses in one examination slot.

The problem is solved using the Hill Climbing local search algorithm.

## State Representation

A timetable is represented as an array:

`[slot(C1), slot(C2), slot(C3), slot(C4), slot(C5), slot(C6)]`

The index represents a course and the value represents its assigned examination slot.

For example:

`[1, 1, 2, 3, 4, 2]`

means:

- C1 → Slot 1
- C2 → Slot 1
- C3 → Slot 2
- C4 → Slot 3
- C5 → Slot 4
- C6 → Slot 2

## Cost Function

The total timetable cost is:

`Total Cost = Conflict Penalty + Distribution Penalty`

Each conflicting course pair scheduled in the same slot produces a penalty of 10.

Therefore:

`Conflict Penalty = Number of Conflicts × 10`

A maximum of two courses can be placed in one slot without a distribution penalty.

For every course beyond two courses in a slot:

`Penalty = 2`

Thus:

`Distribution Penalty = Total Excess Courses × 2`

The objective is to minimize the total cost.

The ideal timetable has:

`Total Cost = 0`

## Neighborhood Function

A neighboring timetable is generated by changing the examination slot of exactly one course.

With 6 courses and 4 slots:

`Number of Neighbors = N × (S - 1)`

`= 6 × 3 = 18`

Therefore, each state has 18 possible neighboring states.

## Hill Climbing

The algorithm works as follows:

1. Start with the initial timetable.
2. Calculate its cost.
3. Generate all neighboring timetables.
4. Calculate the cost of every neighbor.
5. Select the neighbor with the lowest cost.
6. If several neighbors have the same cost, choose the first one generated.
7. Move to the neighbor only when its cost is strictly lower.
8. Repeat until no better neighbor exists.

The final state is a local optimum.

In [12]:
import time

In [13]:
courses = [
    "C1",  # Artificial Intelligence
    "C2",  # Machine Learning
    "C3",  # Data Structures
    "C4",  # Database Systems
    "C5",  # Computer Networks
    "C6"   # Operating Systems
]

N = 6   # Number of courses
S = 4   # Number of examination slots

In [14]:
conflicts = [
    (1, 2),
    (1, 3),
    (2, 4),
    (2, 5),
    (3, 4),
    (3, 6),
    (4, 5),
    (5, 6)
]

K = len(conflicts)

print("Number of conflict pairs:", K)
print("Conflict pairs:", conflicts)

Number of conflict pairs: 8
Conflict pairs: [(1, 2), (1, 3), (2, 4), (2, 5), (3, 4), (3, 6), (4, 5), (5, 6)]


In [15]:
def calculate_cost(state):
    """
    state[i] = examination slot assigned to course Ci+1
    """

    # 1. Calculate course conflicts
    conflict_count = 0

    for c1, c2 in conflicts:
        i = c1 - 1
        j = c2 - 1

        if state[i] == state[j]:
            conflict_count += 1

    conflict_penalty = conflict_count * 10

    # 2. Calculate distribution penalty
    slot_counts = [0] * S

    for slot in state:
        slot_counts[slot - 1] += 1

    excess_courses = 0

    for count in slot_counts:
        excess_courses += max(0, count - 2)

    distribution_penalty = excess_courses * 2

    # 3. Total cost
    total_cost = conflict_penalty + distribution_penalty

    return {
        "conflicts": conflict_count,
        "conflict_penalty": conflict_penalty,
        "slot_counts": slot_counts,
        "excess_courses": excess_courses,
        "distribution_penalty": distribution_penalty,
        "total_cost": total_cost
    }

In [16]:
initial_state = [1, 1, 2, 3, 4, 2]

result = calculate_cost(initial_state)

print("Initial State:", initial_state)
print("Number of Conflicts:", result["conflicts"])
print("Conflict Penalty:", result["conflict_penalty"])
print("Slot Counts:", result["slot_counts"])
print("Excess Courses:", result["excess_courses"])
print("Distribution Penalty:", result["distribution_penalty"])
print("Total Cost:", result["total_cost"])

Initial State: [1, 1, 2, 3, 4, 2]
Number of Conflicts: 2
Conflict Penalty: 20
Slot Counts: [2, 2, 1, 1]
Excess Courses: 0
Distribution Penalty: 0
Total Cost: 20


In [17]:
def generate_neighbors(state):
    """
    Generate all neighboring states by changing
    the slot of exactly one course.
    """

    neighbors = []

    for course_index in range(N):

        current_slot = state[course_index]

        for new_slot in range(1, S + 1):

            if new_slot != current_slot:
                new_state = state.copy()
                new_state[course_index] = new_slot

                neighbors.append(new_state)

    return neighbors

In [18]:
neighbors = generate_neighbors(initial_state)

print("Current State:", initial_state)
print("Number of Neighbors:", len(neighbors))

for i, neighbor in enumerate(neighbors, start=1):
    print(i, neighbor)

Current State: [1, 1, 2, 3, 4, 2]
Number of Neighbors: 18
1 [2, 1, 2, 3, 4, 2]
2 [3, 1, 2, 3, 4, 2]
3 [4, 1, 2, 3, 4, 2]
4 [1, 2, 2, 3, 4, 2]
5 [1, 3, 2, 3, 4, 2]
6 [1, 4, 2, 3, 4, 2]
7 [1, 1, 1, 3, 4, 2]
8 [1, 1, 3, 3, 4, 2]
9 [1, 1, 4, 3, 4, 2]
10 [1, 1, 2, 1, 4, 2]
11 [1, 1, 2, 2, 4, 2]
12 [1, 1, 2, 4, 4, 2]
13 [1, 1, 2, 3, 1, 2]
14 [1, 1, 2, 3, 2, 2]
15 [1, 1, 2, 3, 3, 2]
16 [1, 1, 2, 3, 4, 1]
17 [1, 1, 2, 3, 4, 3]
18 [1, 1, 2, 3, 4, 4]


In [19]:
def print_timetable(state):
    for i in range(N):
        print(f"{courses[i]} -> Slot {state[i]}")

In [20]:
print_timetable(initial_state)

C1 -> Slot 1
C2 -> Slot 1
C3 -> Slot 2
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2


In [21]:
def print_cost_details(state):
    result = calculate_cost(state)

    print("Conflict Cost      =", result["conflict_penalty"])
    print("Distribution Cost  =", result["distribution_penalty"])
    print("Total Cost         =", result["total_cost"])

In [22]:
def hill_climbing(initial_state):

    current_state = initial_state.copy()
    current_cost = calculate_cost(current_state)["total_cost"]

    iteration = 0
    total_neighbors_evaluated = 0

    start_time = time.perf_counter()

    print("=" * 50)
    print("Algorithm: Hill Climbing")
    print("Problem: Exam Timetable Optimization")
    print("=" * 50)

    print("\nInitial Timetable:")
    print_timetable(current_state)

    initial_result = calculate_cost(current_state)

    print("\nInitial Conflict Cost =", initial_result["conflict_penalty"])
    print("Initial Distribution Cost =", initial_result["distribution_penalty"])
    print("Initial Total Cost =", initial_result["total_cost"])

    while True:

        iteration += 1

        # Generate all neighbors
        neighbors = generate_neighbors(current_state)

        # Count all neighbors evaluated
        total_neighbors_evaluated += len(neighbors)

        # Find the best neighbor
        best_neighbor = None
        best_cost = float("inf")

        for neighbor in neighbors:

            neighbor_cost = calculate_cost(neighbor)["total_cost"]

            # Use < so first minimum-cost neighbor is retained
            if neighbor_cost < best_cost:
                best_cost = neighbor_cost
                best_neighbor = neighbor

        print("\n" + "-" * 50)
        print(f"Iteration {iteration}")
        print("Best Neighbor Timetable:")

        print_timetable(best_neighbor)

        best_result = calculate_cost(best_neighbor)

        print("\nConflict Cost =", best_result["conflict_penalty"])
        print("Distribution Cost =", best_result["distribution_penalty"])
        print("Total Cost =", best_result["total_cost"])

        # Stop if no improvement
        if best_cost >= current_cost:
            termination_reason = "Local Optimum"
            break

        # Move to better neighbor
        current_state = best_neighbor
        current_cost = best_cost

    end_time = time.perf_counter()

    execution_time = end_time - start_time

    final_result = calculate_cost(current_state)
    print("Final Timetable:")
    print_timetable(current_state)
    print("\nFinal Conflict Cost =", final_result["conflict_penalty"])
    print("Final Distribution Cost =", final_result["distribution_penalty"])
    print("Final Cost =", final_result["total_cost"])
    print("Iterations =", iteration)
    print("Total Neighboring States Evaluated =", total_neighbors_evaluated)
    print("Execution Time =", execution_time, "seconds")
    print("Termination Reason =", termination_reason)
    return {
        "initial_state": initial_state,
        "initial_cost": calculate_cost(initial_state)["total_cost"],
        "final_state": current_state,
        "final_cost": final_result["total_cost"],
        "iterations": iteration,
        "neighbors_evaluated": total_neighbors_evaluated,
        "execution_time": execution_time,
        "termination_reason": termination_reason
    }

In [23]:
experiment1 = hill_climbing([1, 1, 2, 3, 4, 2])

Algorithm: Hill Climbing
Problem: Exam Timetable Optimization

Initial Timetable:
C1 -> Slot 1
C2 -> Slot 1
C3 -> Slot 2
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Initial Conflict Cost = 20
Initial Distribution Cost = 0
Initial Total Cost = 20

--------------------------------------------------
Iteration 1
Best Neighbor Timetable:
C1 -> Slot 3
C2 -> Slot 1
C3 -> Slot 2
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Conflict Cost = 10
Distribution Cost = 0
Total Cost = 10

--------------------------------------------------
Iteration 2
Best Neighbor Timetable:
C1 -> Slot 3
C2 -> Slot 1
C3 -> Slot 1
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0

--------------------------------------------------
Iteration 3
Best Neighbor Timetable:
C1 -> Slot 2
C2 -> Slot 1
C3 -> Slot 1
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0
Final Timetable:
C1 -> Slot 3
C2 -> Slot 1
C3 -> Slot 1
C4 -> Slot 3
C5 -> Slot 4
C6 ->

In [24]:
initial_state_2 = [2, 3, 1, 1, 4, 4]

experiment2 = hill_climbing(initial_state_2)

Algorithm: Hill Climbing
Problem: Exam Timetable Optimization

Initial Timetable:
C1 -> Slot 2
C2 -> Slot 3
C3 -> Slot 1
C4 -> Slot 1
C5 -> Slot 4
C6 -> Slot 4

Initial Conflict Cost = 20
Initial Distribution Cost = 0
Initial Total Cost = 20

--------------------------------------------------
Iteration 1
Best Neighbor Timetable:
C1 -> Slot 2
C2 -> Slot 3
C3 -> Slot 3
C4 -> Slot 1
C5 -> Slot 4
C6 -> Slot 4

Conflict Cost = 10
Distribution Cost = 0
Total Cost = 10

--------------------------------------------------
Iteration 2
Best Neighbor Timetable:
C1 -> Slot 2
C2 -> Slot 3
C3 -> Slot 3
C4 -> Slot 1
C5 -> Slot 2
C6 -> Slot 4

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0

--------------------------------------------------
Iteration 3
Best Neighbor Timetable:
C1 -> Slot 1
C2 -> Slot 3
C3 -> Slot 3
C4 -> Slot 1
C5 -> Slot 2
C6 -> Slot 4

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0
Final Timetable:
C1 -> Slot 2
C2 -> Slot 3
C3 -> Slot 3
C4 -> Slot 1
C5 -> Slot 2
C6 ->

In [25]:
initial_state_3 = [4, 2, 3, 1, 2, 3]

experiment3 = hill_climbing(initial_state_3)

Algorithm: Hill Climbing
Problem: Exam Timetable Optimization

Initial Timetable:
C1 -> Slot 4
C2 -> Slot 2
C3 -> Slot 3
C4 -> Slot 1
C5 -> Slot 2
C6 -> Slot 3

Initial Conflict Cost = 20
Initial Distribution Cost = 0
Initial Total Cost = 20

--------------------------------------------------
Iteration 1
Best Neighbor Timetable:
C1 -> Slot 4
C2 -> Slot 2
C3 -> Slot 3
C4 -> Slot 1
C5 -> Slot 4
C6 -> Slot 3

Conflict Cost = 10
Distribution Cost = 0
Total Cost = 10

--------------------------------------------------
Iteration 2
Best Neighbor Timetable:
C1 -> Slot 4
C2 -> Slot 2
C3 -> Slot 2
C4 -> Slot 1
C5 -> Slot 4
C6 -> Slot 3

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0

--------------------------------------------------
Iteration 3
Best Neighbor Timetable:
C1 -> Slot 1
C2 -> Slot 2
C3 -> Slot 2
C4 -> Slot 1
C5 -> Slot 4
C6 -> Slot 3

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0
Final Timetable:
C1 -> Slot 4
C2 -> Slot 2
C3 -> Slot 2
C4 -> Slot 1
C5 -> Slot 4
C6 ->

In [26]:
experiments = [
    experiment1,
    experiment2,
    experiment3
]

print("=" * 100)
print(f"{'Exp':<5}{'Initial State':<25}{'Initial Cost':<15}"
      f"{'Final State':<25}{'Final Cost':<15}"
      f"{'Iterations':<12}{'Neighbors':<12}{'Time':<12}")
print("=" * 100)

for i, exp in enumerate(experiments, start=1):

    print(
        f"{i:<5}"
        f"{str(exp['initial_state']):<25}"
        f"{exp['initial_cost']:<15}"
        f"{str(exp['final_state']):<25}"
        f"{exp['final_cost']:<15}"
        f"{exp['iterations']:<12}"
        f"{exp['neighbors_evaluated']:<12}"
        f"{exp['execution_time']:.6f}"
    )

Exp  Initial State            Initial Cost   Final State              Final Cost     Iterations  Neighbors   Time        
1    [1, 1, 2, 3, 4, 2]       20             [3, 1, 1, 3, 4, 2]       0              3           54          0.000453
2    [2, 3, 1, 1, 4, 4]       20             [2, 3, 3, 1, 2, 4]       0              3           54          0.000738
3    [4, 2, 3, 1, 2, 3]       20             [4, 2, 2, 1, 4, 3]       0              3           54          0.000472


In [27]:
best_experiment = min(
    experiments,
    key=lambda x: x["final_cost"]
)

print("Best Initial State :", best_experiment["initial_state"])
print("Best Final State   :", best_experiment["final_state"])
print("Best Final Cost    :", best_experiment["final_cost"])

Best Initial State : [1, 1, 2, 3, 4, 2]
Best Final State   : [3, 1, 1, 3, 4, 2]
Best Final Cost    : 0


## PS-4 Experimental Analysis

Three different initial timetables were tested to observe the effect of the initial state on Hill Climbing.

### Experiment 1

Initial state:

`[1, 1, 2, 3, 4, 2]`

Initial cost = `20`

The initial cost contains two conflicting course pairs:

- C1–C2
- C3–C6

The distribution penalty is 0 because no slot contains more than two courses.

The Hill Climbing algorithm improves the timetable until it reaches a state with total cost 0.

### Experiment 2

Initial state:

`[2, 3, 1, 1, 4, 4]`

Initial cost = `20`

The algorithm again searches neighboring states and reaches a timetable with total cost 0.

### Experiment 3

Initial state:

`[4, 2, 3, 1, 2, 3]`

Initial cost = `20`

Hill Climbing again improves the timetable and reaches a solution with total cost 0.

## Comparison

All three initial states have an initial cost of 20 and can reach a timetable with cost 0.

The final timetable may differ between experiments because Hill Climbing is a local search method and its result depends on the starting state and the order in which neighbors are generated.

A cost of 0 is the ideal solution because it has both zero conflict penalty and zero distribution penalty.

Therefore, the experiments demonstrate the effect of the initial state on the search path, number of iterations, and neighboring states evaluated.